Cohere Labs

# Cohere Labs · MuseumSCAT — Competition Journal

*Community notebook in the Cohere Labs Open Science spirit — not an official Cohere publication.*

| | |
|---|---|
| **Task** | transcribe `verbatimDate` + `verbatimLocality` from 8192×5464 specimen tray photographs |
| **Metric** | `mean(AURC_date, AURC_locality)`, lower is better — **rank-only**: the confidence scale is free |
| **Submission** | replay of the 1st-place sheet, **0.00478** (written in §3, before any GPU work) |
| **Experiment** | TrOCR-base fine-tuned on the 200 labelled specimens, **T4 ×1, 6 h budget** |
| **Question** | how far does conventional supervised OCR get, and is its own likelihood a better confidence than a hand-built risk model? |
| **Honest expectation** | the fine-tune will **not** approach 0.00478 — 200 labelled plates is a very small corpus |

---

### How to read this notebook

It has **two arms that never touch each other**.

The **submission arm** (§1–§4) writes `submission.csv` and validates it. It runs first, on CPU,
in seconds. Nothing downstream can modify it — that is deliberate, and it is the single most
useful habit this competition taught me.

The **journal arm** (§5–§13) is the write-up: the metric, the ranking layer that actually won,
five figures, and a genuine 6-hour training experiment on T4. If the GPU work fails, times out,
or the images are not attached, the submission is already on disk and still correct.

> **Where the score came from.** The transcriptions were produced by multi-pass vision-model
> reading of all 3,300 plates — roughly 6M tokens — which cannot re-run inside a kernel. What
> *is* fully reproducible here, and what the leaderboard actually rewarded, is the **ranking
> layer**: text accuracy never changed between 0.00584 and 0.00478. Every one of those points
> came from re-ordering the same strings.


## 1. Time budget and environment

Six hours, allocated up front. The submission and the journal figures are cheap; the training
run gets whatever is left after a reserve, and stops on the clock rather than on an epoch count.


In [ ]:
import os, re, csv, glob, sys, time, json, math
import numpy as np

T_START = time.time()
TOTAL_BUDGET_S = 6 * 3600          # the session budget we were given
RESERVE_S      = 25 * 60           # kept back for evaluation, figures and the final re-check

IS_RERUN = os.getenv("KAGGLE_IS_COMPETITION_RERUN") is not None

def elapsed():   return time.time() - T_START
def remaining(): return TOTAL_BUDGET_S - elapsed()

try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else "none"
except Exception:
    torch, HAS_GPU, GPU_NAME = None, False, "torch unavailable"

print(f"budget    : {TOTAL_BUDGET_S/3600:.1f} h   reserve {RESERVE_S/60:.0f} min")
print(f"gpu       : {GPU_NAME}")
print(f"rerun     : {IS_RERUN}")


## 2. Inputs

We take the first CSV under `/kaggle/input` that carries the three required columns. Two decoys
have to be filtered out by hand: attaching the competition data source drops `train.csv`
(200 labelled rows) and a `sample_submission` next to the predictions, and **both** carry those
columns. A naive first-match search silently emits a 200-row or all-placeholder sheet.


In [ ]:
REQUIRED = ("image_file", "verbatimDate", "verbatimLocality")
CONF     = ("verbatimDate_confidence", "verbatimLocality_confidence")
COLS     = ["image_file", "verbatimDate", "verbatimDate_confidence",
            "verbatimLocality", "verbatimLocality_confidence"]

SEARCH = ["/kaggle/input/**/*.csv", "E:/Claude code/museumscat/deliver/*.csv", "./*.csv"]
SKIP   = ("train.csv", "sample_submission", "test.csv")
PREFER = "sub_m0.15"
N_TEST = 3300


def read_csv(path):
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


def write_csv(path, rows, cols):
    with open(path, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=cols, lineterminator="\n")
        w.writeheader()
        for r in rows:
            w.writerow({c: r[c] for c in cols})


def _rank_candidate(rows, path, has_conf):
    """Higher is better: the named file first, then real confidences, then full length."""
    named = 2 if (PREFER and PREFER.lower() in os.path.basename(path).lower()) else 0
    spread = 1 if has_conf and len({r[CONF[1]] for r in rows}) > len(rows) // 2 else 0
    return (named, int(has_conf) + spread, 1 if len(rows) >= N_TEST else 0)


def find_predictions():
    best, seen = None, 0
    for pat in SEARCH:
        for p in sorted(glob.glob(pat, recursive=True)):
            if any(s in os.path.basename(p).lower() for s in SKIP):
                continue
            try:
                rows = read_csv(p)
            except Exception:
                continue
            if not rows or not all(c in rows[0] for c in REQUIRED):
                continue
            seen += 1
            hc = all(c in rows[0] for c in CONF)
            cand = (_rank_candidate(rows, p, hc), rows, p, hc)
            if best is None or cand[0] > best[0]:
                best = cand
    print(f"usable candidates : {seen}")
    return (None, None, False) if best is None else (best[1], best[2], best[3])


ROWS, SRC, HAS_CONF = find_predictions()
print("source     :", SRC)
print("rows       :", 0 if ROWS is None else len(ROWS))
print("has conf   :", HAS_CONF)
if ROWS and len(ROWS) != N_TEST:
    print(f"WARNING: expected {N_TEST} test rows, got {len(ROWS)}")


## 3. The submission — written first, on purpose

**This is the only cell that produces the scored artefact.** It runs before the metric is even
defined, before any figure, and long before the GPU is touched.

That ordering is a lesson from this competition rather than a stylistic choice. An earlier
attempt read the test set with two reader passes launched as one long job; the session hit its
limit at roughly 114 of 264 batches and there was no complete CSV to show for it. Since then
the rule has been: **produce a valid submission first, improve it second.**

If the input already carries confidences we replay it unchanged. Otherwise we rank the text with
the layer in §8 and emit fresh confidences.


In [ ]:
SUBMISSION = "submission.csv"
assert ROWS, "no prediction CSV found - attach the predictions dataset"

if HAS_CONF:
    SUB_MODE, out = "replay", ROWS
else:
    SUB_MODE = "rank-from-text"          # filled in by section 8 if reached
    out = None

if out is not None:
    write_csv(SUBMISSION, out, COLS)
    print(f"mode={SUB_MODE}   wrote {SUBMISSION} with {len(out)} rows at {elapsed():.0f}s")
else:
    print("text-only input: confidences will be built in section 8")


## 4. Validating the submission

Structural checks that have each caught a real bug. **Ties matter more than they look**: AURC
reads only the induced order, so tied confidences quietly hand the tie-break to whatever order
the file happens to be written in.


In [ ]:
def validate(path=SUBMISSION, expect=N_TEST):
    sub = read_csv(path)
    ids = [r["image_file"] for r in sub]
    dc = np.array([float(r["verbatimDate_confidence"]) for r in sub])
    lc = np.array([float(r["verbatimLocality_confidence"]) for r in sub])
    checks = {
        f"row count {expect}"   : len(sub) == expect,
        "unique image_file"     : len(set(ids)) == len(ids),
        "confidences in [0,1]"  : bool(dc.min() >= 0 and dc.max() <= 1
                                       and lc.min() >= 0 and lc.max() <= 1),
        "no tied date conf"     : len(set(dc.tolist())) == len(dc),
        "no tied locality conf" : len(set(lc.tolist())) == len(lc),
        "no empty cells"        : all(r["verbatimDate"] and r["verbatimLocality"] for r in sub),
    }
    for k, v in checks.items():
        print(f"  {'PASS' if v else 'FAIL'}  {k}")
    return sub, dc, lc, all(checks.values())


if os.path.exists(SUBMISSION):
    SUB, DC, LC, SUB_OK = validate()
    miss = [i for i, r in enumerate(SUB) if r["verbatimLocality"].upper() == "MISSING"]
    order = {i: k for k, i in enumerate(np.argsort(-LC))}
    print(f"\nMISSING localities: {len(miss)}, median rank "
          f"{int(np.median([order[i] for i in miss]))} of {len(SUB)}   (0 = most confident)")
    print(f"submission ready at {elapsed():.0f}s - everything below is the journal")


## 5. The metric

`AURC` averages the *running mean error* over every coverage level. An error in position 1 is
counted into nearly all `n` of those means; an error in the last position is counted once. That
asymmetry is the entire game, and it is why this notebook spends far more effort on ordering
than on transcription.


In [ ]:
%%writefile mscat_metric.py
# -*- coding: utf-8 -*-
"""MuseumSCAT scoring: normalised edit distance + Area Under the Risk-Coverage curve.

The competition ranks you on mean(AURC_date, AURC_locality), lower is better.
AURC only ever looks at the *order* your confidences induce, never their scale.
"""
import re
from itertools import permutations

import numpy as np

# Date punctuation the organisers treat as interchangeable: . , - middot / space
_DATE_SEP = re.compile(r"[.,\-\u00b7/\s]+")


def levenshtein(a: str, b: str) -> int:
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def ned(a: str, b: str) -> float:
    """Normalised edit distance, case-insensitive. 0.0 = identical, 1.0 = nothing in common."""
    a, b = (a or "").strip().lower(), (b or "").strip().lower()
    if a == "missing":
        a = ""
    if b == "missing":
        b = ""
    if not a and not b:
        return 0.0
    return levenshtein(a, b) / max(len(a), len(b))


def ned_date(a: str, b: str) -> float:
    """Date NED, after collapsing the interchangeable separators to a single space."""
    norm = lambda s: _DATE_SEP.sub(" ", (s or "").strip().lower()).strip()
    return ned(norm(a), norm(b))


def ned_cards(pred: str, gt: str, date: bool = False) -> float:
    """Multi-card fields are pipe-separated and matched over every ordering."""
    f = ned_date if date else ned
    p_cards = [c.strip() for c in (pred or "").split("|")]
    g_cards = [c.strip() for c in (gt or "").split("|")]
    if len(g_cards) == 1 and len(p_cards) == 1:
        return f(pred, gt)
    best = 1.0
    for perm in permutations(g_cards):
        best = min(best, f(" | ".join(p_cards), " | ".join(perm)))
    return best


def aurc(errors, confidence) -> float:
    """Area under the risk-coverage curve.

    Sort rows by confidence descending, then average the running mean error over
    every coverage level k = 1..n. An error in the first few positions is counted
    into almost every one of the n running means; an error in the last position is
    counted once. That asymmetry is the whole game.
    """
    e = np.asarray(errors, dtype=float)
    c = np.asarray(confidence, dtype=float)
    order = np.argsort(-c, kind="stable")
    e = e[order]
    k = np.arange(1, len(e) + 1)
    return float(np.mean(np.cumsum(e) / k))


def score(pred_rows, gt_rows) -> dict:
    """Full competition score: the mean of the two per-field AURCs."""
    gt = {r["image_file"]: r for r in gt_rows}
    keys = [r["image_file"] for r in pred_rows if r["image_file"] in gt]
    d_err = [ned_cards(r["verbatimDate"], gt[r["image_file"]]["verbatimDate"], date=True)
             for r in pred_rows if r["image_file"] in gt]
    l_err = [ned_cards(r["verbatimLocality"], gt[r["image_file"]]["verbatimLocality"])
             for r in pred_rows if r["image_file"] in gt]
    d_cf = [float(r["verbatimDate_confidence"]) for r in pred_rows if r["image_file"] in gt]
    l_cf = [float(r["verbatimLocality_confidence"]) for r in pred_rows if r["image_file"] in gt]
    d, l = aurc(d_err, d_cf), aurc(l_err, l_cf)
    return {"n": len(keys), "AURC_date": d, "AURC_locality": l, "score": (d + l) / 2,
            "meanNED_date": float(np.mean(d_err)), "meanNED_locality": float(np.mean(l_err))}


In [ ]:
from mscat_metric import ned, ned_date, ned_cards, aurc, score

assert ned("Kb", "kb") == 0.0                          # case-insensitive
assert ned("MISSING", "") == 0.0                       # MISSING is the empty string
assert ned_date("27.IV.2022", "27 IV 2022") == 0.0     # punctuation equivalence
assert ned_cards("A | B", "B | A") == 0.0              # multi-card order is free
assert aurc([0.0] * 100, list(range(100))) == 0.0      # a perfect sheet scores 0
print("metric self-checks passed")


## 6. Figure 1 — why ordering dominates

Ninety perfect rows, ten completely wrong ones, and **nothing changed but the order**. AURC is
the area under each curve.


In [ ]:
%%writefile mscat_viz.py
# -*- coding: utf-8 -*-
"""Journal figures for the MuseumSCAT write-up.

One house style, applied everywhere: thin marks, hairline solid grid, a legend whenever
two or more series share a panel, and selective direct labels rather than a number on
every point. Colours come from a validated categorical palette, assigned in fixed slot
order; de-emphasised series use ink grey, never a fourth hue.
"""
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# --- palette (validated categorical slots 1-3, light mode) -----------------
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_2 = "#52514e"
GRID = "#e8e7e4"
MUTED = "#b8b7b3"
S1, S2, S3 = "#2a78d6", "#eb6834", "#1baf7a"   # blue, orange, aqua


def style():
    matplotlib.rcParams.update({
        "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE, "figure.dpi": 130,
        "font.family": "DejaVu Sans", "font.size": 10,
        "text.color": INK, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
        "xtick.color": INK_2, "ytick.color": INK_2,
        "axes.edgecolor": GRID, "axes.linewidth": 0.8,
        "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.7,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "legend.frameon": False, "figure.autolayout": True,
    })


def _frame(ax, title=None, xlabel=None, ylabel=None, ygrid=True):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.grid(axis="y" if ygrid else "x")
    ax.grid(axis="x" if ygrid else "y", visible=False)
    if title:
        ax.set_title(title, loc="left", fontsize=12, pad=12, color=INK)
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=9)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=9)
    return ax


# ------------------------------------------------------------------ fig 1
def risk_coverage(errors, orderings, labels, title=None, save=None):
    """Risk-coverage curves. AURC is literally the area under each of these.

    The panel exists to show that the same predictions trace wildly different curves
    depending only on the order they are read in.
    """
    style()
    fig, ax = plt.subplots(figsize=(7.6, 4.3))
    colors = [S1, MUTED, S2]
    e = np.asarray(errors, float)
    n = len(e)
    cov = np.arange(1, n + 1) / n
    # No endpoint labels: every curve necessarily meets at coverage 1.0 (full coverage is
    # just the mean error), so direct labels there would collide and say nothing. The
    # legend carries identity and the AURC each curve integrates to.
    for i, (order, lab) in enumerate(zip(orderings, labels)):
        risk = np.cumsum(e[order]) / np.arange(1, n + 1)
        ax.plot(cov, risk, lw=2.0, color=colors[i % 3], label=f"{lab}  (AURC {risk.mean():.4f})")
    _frame(ax, title or "Risk-coverage: identical predictions, three orderings",
           "coverage (fraction of rows kept, most confident first)", "risk (mean NED)")
    ax.set_xlim(0, 1.01)
    ax.legend(loc="upper right", fontsize=8.5)
    if save:
        fig.savefig(save, bbox_inches="tight")
    return fig


# ------------------------------------------------------------------ fig 2
def length_prior(localities, conf, title=None, save=None):
    """Mean confidence rank against locality length - the length prior, visible."""
    style()
    L = np.array([0 if s.upper() == "MISSING" else len(s) for s in localities])
    rank = np.empty(len(conf), int)
    rank[np.argsort(-np.asarray(conf, float), kind="stable")] = np.arange(len(conf))
    pct = 100.0 * rank / (len(conf) - 1)

    edges = [1, 5, 10, 15, 20, 25, 30, 40, 60, 200]
    xs, ys, ns = [], [], []
    for a, b in zip(edges[:-1], edges[1:]):
        m = (L >= a) & (L < b)
        if m.sum() >= 15:
            xs.append(f"{a}-{b - 1}")
            ys.append(pct[m].mean())
            ns.append(int(m.sum()))

    fig, ax = plt.subplots(figsize=(7.6, 4.0))
    ax.bar(xs, ys, color=S1, width=0.62)
    for x, y, c in zip(xs, ys, ns):
        ax.annotate(f"n={c}", (x, y), xytext=(0, 4), textcoords="offset points",
                    ha="center", fontsize=7.5, color=INK_2)
    _frame(ax, title or "Longer localities are ranked less confident",
           "locality length (characters, MISSING excluded)", "mean rank position (%, 0 = most confident)")
    ax.set_ylim(0, 100)
    if save:
        fig.savefig(save, bbox_inches="tight")
    return fig


# ------------------------------------------------------------------ fig 3
def missing_placement(rank_before, rank_after, n, title=None, save=None):
    """Where MISSING localities sit in the ranking, before and after the penalty."""
    style()
    fig, ax = plt.subplots(figsize=(7.6, 4.0))
    bins = np.linspace(0, 100, 26)
    ax.hist(100.0 * np.asarray(rank_before) / (n - 1), bins=bins, color=MUTED,
            label="w_miss = 0     (all at the top, where errors cost most)")
    ax.hist(100.0 * np.asarray(rank_after) / (n - 1), bins=bins, color=S1, alpha=0.92,
            label="w_miss = 0.15  (shipped)")
    _frame(ax, title or "The MISSING block, moved off the top of the ranking",
           "rank position (%, 0 = most confident)", "MISSING localities")
    ax.set_ylim(0, ax.get_ylim()[1] * 1.34)      # headroom so the legend clears the bars
    ax.legend(loc="upper right", fontsize=8.5)
    if save:
        fig.savefig(save, bbox_inches="tight")
    return fig


# ------------------------------------------------------------------ fig 4
def weight_sweep(weights, scores, best_idx, title=None, save=None):
    """A one-parameter sweep. Emphasis, not eight hues: the optimum is the story."""
    style()
    fig, ax = plt.subplots(figsize=(7.6, 4.0))
    ax.plot(weights, scores, lw=2.0, color=S1, marker="o", ms=5,
            markerfacecolor=SURFACE, markeredgewidth=1.6, markeredgecolor=S1)
    bx, by = weights[best_idx], scores[best_idx]
    ax.plot([bx], [by], marker="o", ms=9, color=S2, zorder=5)
    # Open a clear band beneath the curve so the callout cannot land on the tick labels.
    lo, hi = min(scores), max(scores)
    ax.set_ylim(lo - 0.22 * (hi - lo), hi + 0.08 * (hi - lo))
    ax.annotate(f"{by:.5f}  at w_miss={bx}", (bx, by), xytext=(10, -18),
                textcoords="offset points", fontsize=9, color=INK, fontweight="bold")
    _frame(ax, title or "MISSING-locality penalty: leaderboard sweep",
           "w_miss", "leaderboard score (lower is better)")
    if save:
        fig.savefig(save, bbox_inches="tight")
    return fig


# ------------------------------------------------------------------ fig 5
def ladder(labels, scores, title=None, save=None):
    """The leaderboard ladder. Log scale, because the span is a full order of magnitude."""
    style()
    fig, ax = plt.subplots(figsize=(7.8, 5.4))
    y = np.arange(len(labels))[::-1]
    colors = [MUTED] * len(labels)
    colors[int(np.argmin(scores))] = S1
    ax.barh(y, scores, color=colors, height=0.62)
    ax.set_yticks(y, labels, fontsize=8.6)
    ax.set_xscale("log")
    for yy, s in zip(y, scores):
        ax.annotate(f"{s:.5f}", (s, yy), xytext=(5, 0), textcoords="offset points",
                    va="center", fontsize=8.2, color=INK_2)
    _frame(ax, title or "Leaderboard ladder (log scale)", "score (lower is better)",
           None, ygrid=False)
    ax.set_xlim(min(scores) * 0.72, max(scores) * 1.5)
    if save:
        fig.savefig(save, bbox_inches="tight")
    return fig


# ------------------------------------------------------------------ fig 6
def error_vs_length(lengths, neds, title=None, save=None):
    """Ground-truth check of the length prior: does error really rise with length?"""
    style()
    L, E = np.asarray(lengths, float), np.asarray(neds, float)
    edges = [1, 5, 10, 15, 20, 25, 30, 40, 200]
    xs, ys, ns = [], [], []
    for a, b in zip(edges[:-1], edges[1:]):
        m = (L >= a) & (L < b)
        if m.sum() >= 5:
            xs.append(f"{a}-{b - 1}")
            ys.append(float(E[m].mean()))
            ns.append(int(m.sum()))
    fig, ax = plt.subplots(figsize=(7.6, 4.0))
    ax.bar(xs, ys, color=S3, width=0.62)
    for x, y, c in zip(xs, ys, ns):
        ax.annotate(f"n={c}", (x, y), xytext=(0, 4), textcoords="offset points",
                    ha="center", fontsize=7.5, color=INK_2)
    _frame(ax, title or "Measured error against locality length (labelled rows)",
           "locality length (characters)", "mean NED")
    if save:
        fig.savefig(save, bbox_inches="tight")
    return fig


In [ ]:
import matplotlib.pyplot as plt
import mscat_viz as V

rs = np.random.RandomState(0)
demo_err = np.array([0.0] * 90 + [1.0] * 10)
best  = np.argsort(demo_err, kind="stable")
worst = best[::-1]
rand  = rs.permutation(len(demo_err))

fig = V.risk_coverage(demo_err, [best, rand, worst], ["perfect", "random", "worst"])
plt.show()

a_best, a_worst = aurc(demo_err, -np.argsort(best)), aurc(demo_err, -np.argsort(worst))
print(f"perfect ordering AURC {a_best:.5f}   worst ordering AURC {a_worst:.5f}"
      f"   ->  {a_worst / a_best:.0f}x spread on identical predictions")


## 7. Transcription conventions

Measured on the 200 labelled rows, then confirmed on the leaderboard: this module alone moved a
reproduced public baseline from **0.04462 to 0.04305** without changing a single reader output.

The rule that pays for itself is `strip_provenance`. *"Mus. Løvendal"* is a collection marker,
not a place — it appears in **zero** of the 200 ground-truth localities, yet readers keep
appending it to real ones. Worth **0.00609 → 0.00596** by itself. A systematic hunt for other
suffixes with the same signature found none: Rønne, Bornholm, Sjælland and Klint are all genuine
regional qualifiers, and stripping them would have destroyed real answers.


In [ ]:
%%writefile mscat_text.py
# -*- coding: utf-8 -*-
"""Transcription conventions for the NHM Denmark dung-beetle labels.

Every rule here was measured against the 200 labelled training rows and then
confirmed on the leaderboard: applying this module alone moved the public C157
baseline from 0.04462 to 0.04305 without changing a single reader.
"""
import re

# "Mus. L\u00f8vendal" is a provenance/collection marker, not a place. It never appears in
# any of the 200 ground-truth localities, yet readers happily append it to real ones.
_PROVENANCE = {"lovendal", "l\u00f8vendal", "l\u00f6vendal", "lyvendal", "loevendal"}

# "Dania" is the latinised country name printed on the card stock; never a locality.
_DANIA = re.compile(r"^\s*dania\s*[:,\-]?\s*", re.IGNORECASE)

_QUOTES = {
    "\u201e": ",,",   # low double quote, printed as a double comma on these labels
    "\u201c": "''", "\u201d": "''", "\u2018": "'", "\u2019": "'",
}


def fix_chars(s: str) -> str:
    """Character-level conventions the ground truth is consistent about."""
    s = s.replace("\u00ff", "y")          # y-diaeresis is transcribed as plain y (3 GT cases)
    for k, v in _QUOTES.items():
        s = s.replace(k, v)
    return s


def strip_provenance(card: str) -> str:
    """Drop collection-marker tokens, but never empty a card and never touch punctuation."""
    words = card.split()
    kept = [w for w in words if w.lower().strip(".,;:") not in _PROVENANCE]
    if len(kept) == len(words) or not kept:
        return card                       # nothing removed: leave the card exactly as read
    return " ".join(kept).strip(" ,")


def strip_trailing_period(s: str) -> str:
    """A trailing period is real abbreviation ("Bovbj.") on short words, noise on long ones."""
    if not s.endswith("."):
        return s
    last = s[:-1].split()[-1] if s[:-1].split() else ""
    return s[:-1] if len(last) >= 6 else s


def clean_locality(s: str) -> str:
    if not s or s.upper() == "MISSING":
        return "MISSING"
    cards = [strip_provenance(_DANIA.sub("", fix_chars(c.strip()))) for c in s.split("|")]
    cards = [strip_trailing_period(c.strip()) for c in cards if c.strip()]
    return " | ".join(cards) if cards else "MISSING"


def clean_date(s: str) -> str:
    if not s or s.upper() == "MISSING":
        return "MISSING"
    return " | ".join(fix_chars(c.strip()) for c in s.split("|") if c.strip()) or "MISSING"


# --- date sanitiser -------------------------------------------------------
# Small cards holding only 2-3 digits are catalogue numbers in this collection, not
# dates. Promoting them cost 0.00833 -> 0.00992 on the leaderboard, so we reject them.
_BARE_NUM = re.compile(r"^\d{2,3}\.?$")
_DATE_WORD = re.compile(
    r"(i|ii|iii|iv|v|vi|vii|viii|ix|x|xi|xii|jan|feb|mar|apr|maj|may|jun|jul|aug|sep|okt|"
    r"oct|nov|dec|primo|medio|ultimo)")


def is_bad_date(candidate: str, original_was_missing: bool) -> bool:
    """True when a proposed date override should be rejected."""
    c = candidate.strip()
    if _BARE_NUM.match(c):
        return True
    if original_was_missing and all(_BARE_NUM.match(p.strip()) for p in c.split("|")):
        return True
    if re.search(r"[a-zA-Z]", c) and not _DATE_WORD.search(c.lower()):
        return True                       # letters that are not a month name = misread card
    return False


In [ ]:
from mscat_text import clean_locality, clean_date, is_bad_date

for s in ["Rold skov L\u00f8vendal", "Bovbj.", "Dronninglund.", "Dania: Jylland", "L\u00f8vendal"]:
    print(f"{s!r:32} -> {clean_locality(s)!r}")

# Bare 2-3 digit cards are catalogue numbers in this collection, not dates.
# Promoting them to dates cost 0.00833 -> 0.00992, so the sanitiser rejects them.
print()
for c in ["495", "27.IV.2022", "Coll. Hansen"]:
    print(f"is_bad_date({c!r:14}) = {is_bad_date(c, False)}")


## 8. The ranking layer

Additive risk, where every term answers one question: *which rows sit near the top of the
ranking without having earned it?*


In [ ]:
%%writefile mscat_rank.py
# -*- coding: utf-8 -*-
"""The AURC-aware ranking layer.

This is the part of the solution that actually moved the leaderboard. Text accuracy
stayed flat from 0.00584 down to 0.00478; every one of those points came from
re-ordering the same predictions.

Risk is additive. Each term answers one question: "which rows are sitting near the
top of the ranking without having earned it?"
"""
import re
import unicodedata
from collections import Counter

import numpy as np

# Empirical error rates measured on the 200 labelled rows, keyed by (reader grade, is_missing).
# Note the inversion in the locality table: grade-B localities beat grade-A ones. Readers
# label a crop "A" when it is *legible*, not when it is *easy*, and the long legible ones
# are exactly where transcription drifts.
DATE_RISK = {("A", True): 0.000, ("A", False): 0.000, ("B", True): 0.005, ("B", False): 0.010,
             ("C", True): 0.040, ("C", False): 0.079, ("D", True): 0.120, ("D", False): 0.200}
LOC_RISK = {("A", True): 0.000, ("B", True): 0.004, ("B", False): 0.007, ("A", False): 0.011,
            ("C", True): 0.030, ("C", False): 0.049, ("D", True): 0.150, ("D", False): 0.277}

# Leaderboard-tuned weights for the locality field (the 0.00478 configuration).
W_LEN = 0.011      # per character, after capping
LEN_CAP = 22.0     # beyond ~22 chars, extra length stops predicting extra error
W_MISS = 0.15      # flat penalty on MISSING localities  <- the single biggest win
W_RARE = 0.05      # self-consistency gazetteer rarity
W_AB = 0.12        # the two independent readers disagree
W_ANC = 0.12       # we disagree with the external anchor reader
W_D4 = 6.00        # the third-look arbiter disagrees
BLOCK_DISCOUNT = 1.0   # length discount for pipe/coordinate rows (fitted to 15 rows => kept off)

_COORD = re.compile(r"\d+[.,]\d+\s*[NSEW]")


def is_missing(s: str) -> bool:
    return (not s) or s.strip().upper() == "MISSING"


def norm_key(s: str) -> str:
    s = unicodedata.normalize("NFKD", s.lower())
    s = "".join(c for c in s if not unicodedata.combining(c))
    return " ".join(re.sub(r"[^a-z0-9 ]", " ", s).split())


def build_gazetteer(localities) -> Counter:
    """Self-consistency gazetteer: token frequencies over our own predictions.

    Entirely label-free. A place name we transcribed the same way 40 times is far more
    likely correct than one that appears once; the second is either genuinely rare or
    a misread, and both deserve to rank lower.
    """
    toks = Counter()
    for s in localities:
        if is_missing(s):
            continue
        for w in norm_key(s).split():
            if len(w) >= 3:
                toks[w] += 1
    return toks


def rarity(s: str, toks: Counter) -> float:
    if is_missing(s):
        return 0.0
    ws = [w for w in norm_key(s).split() if len(w) >= 3]
    if not ws:
        return 0.0
    return float(np.mean([1.0 / np.log2(2 + toks[w]) for w in ws]))


def locality_risk(localities, grades=None, disagree_ab=None, disagree_anchor=None,
                  disagree_third=None, w_len=W_LEN, w_miss=W_MISS, w_rare=W_RARE):
    """Additive locality risk. Only `localities` is required; the rest are optional
    signals from extra reader passes and simply contribute 0 when absent."""
    n = len(localities)
    toks = build_gazetteer(localities)
    miss = np.array([1.0 if is_missing(s) else 0.0 for s in localities])

    # Length prior: error rises monotonically with locality length, then plateaus.
    length = np.array([0.0 if is_missing(s) else float(len(s)) for s in localities])
    length = np.minimum(length, LEN_CAP)
    if BLOCK_DISCOUNT != 1.0:
        blk = np.array([1.0 if ("|" in s or _COORD.search(s)) else 0.0 for s in localities])
        length = np.where(blk > 0, length * BLOCK_DISCOUNT, length)

    risk = w_len * length

    # MISSING localities have length 0, so the length prior alone floats all ~478 of them
    # to the very top of the ranking, where AURC punishes an error hardest. A flat penalty
    # lets short, near-perfect localities ("Kb", "Ti") outrank them instead.
    risk = risk + w_miss * miss
    risk = risk + w_rare * np.array([rarity(s, toks) for s in localities])

    if grades is not None:
        risk = risk + np.array([LOC_RISK.get((g, is_missing(s)), 0.05)
                                for g, s in zip(grades, localities)])
    for signal, weight in ((disagree_ab, W_AB), (disagree_anchor, W_ANC), (disagree_third, W_D4)):
        if signal is not None:
            risk = risk + weight * np.asarray(signal, dtype=float)
    return risk


def date_risk(dates, grades=None, disagree_ab=None):
    """Dates are close to solved; do NOT reuse the locality tricks here.

    A length prior hurt on train (0.00071 -> 0.00109) and a MISSING penalty hurt on the
    leaderboard (0.00478 -> 0.00540): an absent date really is near-certain, because a
    card with no date on it is unambiguous, whereas an absent locality is usually a miss.
    """
    risk = np.zeros(len(dates))
    if grades is not None:
        risk = risk + np.array([DATE_RISK.get((g, is_missing(s)), 0.05)
                                for g, s in zip(grades, dates)])
    if disagree_ab is not None:
        risk = risk + W_AB * np.asarray(disagree_ab, dtype=float)
    return risk


def to_confidence(risk, hi=0.99, lo=0.01):
    """Map risk to confidence by rank.

    AURC reads only the induced ordering, so the numeric scale is free. Spreading the
    values evenly over [lo, hi] guarantees no ties and keeps the column human-readable.
    """
    risk = np.asarray(risk, dtype=float)
    n = len(risk)
    if n == 1:
        return np.array([hi])
    rank = np.empty(n, dtype=int)
    rank[np.argsort(risk, kind="stable")] = np.arange(n)
    return hi - rank * (hi - lo) / (n - 1)


In [ ]:
import mscat_rank as R

if not os.path.exists(SUBMISSION):        # text-only input: build confidences now
    loc = [r["verbatimLocality"] for r in ROWS]
    dat = [r["verbatimDate"] for r in ROWS]
    lc_new = R.to_confidence(R.locality_risk(loc))
    dc_new = R.to_confidence(R.date_risk(dat))
    out = [{"image_file": r["image_file"],
            "verbatimDate": r["verbatimDate"], "verbatimDate_confidence": f"{a:.9f}",
            "verbatimLocality": r["verbatimLocality"], "verbatimLocality_confidence": f"{b:.9f}"}
           for r, a, b in zip(ROWS, dc_new, lc_new)]
    write_csv(SUBMISSION, out, COLS)
    SUB, DC, LC, SUB_OK = validate()
    print(f"mode={SUB_MODE}   wrote {SUBMISSION}")
else:
    print("submission already written in section 3 - ranking layer used for the journal only")


## 9. Figures 2–3 — the length prior, and the discovery that won

Two facts about locality strings, both visible in the shipped ranking.

**Left to itself, the length prior has a hole in it.** `MISSING` has length 0, so all 478 of
them float to the very top — precisely where AURC punishes an error hardest. A flat penalty
pushes them down so short, near-perfect localities (`"Kb"`, `"Ti"`) can outrank them. That one
term took the sheet from 0.00584 to **0.00478** and first place.


In [ ]:
LOCS = [r["verbatimLocality"] for r in SUB]

fig = V.length_prior(LOCS, LC); plt.show()

r0 = R.to_confidence(R.locality_risk(LOCS, w_miss=0.0))
r1 = R.to_confidence(R.locality_risk(LOCS, w_miss=0.15))
n = len(LOCS)
p0 = np.empty(n, int); p0[np.argsort(-r0, kind="stable")] = np.arange(n)
p1 = np.empty(n, int); p1[np.argsort(-r1, kind="stable")] = np.arange(n)
midx = [i for i, s in enumerate(LOCS) if s.upper() == "MISSING"]

fig = V.missing_placement(p0[midx], p1[midx], n); plt.show()
print(f"MISSING median rank   w_miss=0.00 : {100*np.median(p0[midx])/(n-1):5.1f}%")
print(f"MISSING median rank   w_miss=0.15 : {100*np.median(p1[midx])/(n-1):5.1f}%")


## 10. Figures 4–5 — the sweep and the ladder

The sweep is a clean single-minimum curve, which is the main reason I trusted it over local
cross-validation. Note the asymmetry: overshooting to 0.30 costs more than leaving the term out
entirely.

The ladder is every leaderboard-measured step. Everything from 0.00833 downward is **ranking** —
the text stopped changing there.


In [ ]:
W = [0.0, 0.03, 0.08, 0.12, 0.15, 0.20, 0.30]
S = [0.00584, 0.00562, 0.00511, 0.00480, 0.00478, 0.00496, 0.00547]
fig = V.weight_sweep(W, S, int(np.argmin(S))); plt.show()

LADDER = [("public C157 baseline", 0.04462), ("+ text conventions", 0.04305),
          ("readers, 46% coverage", 0.02309), ("full coverage", 0.01302),
          ("+ gazetteer rarity", 0.01187), ("+ dual reader + arbitration", 0.00937),
          ("+ full arbitration + date sanitiser", 0.00833), ("+ length prior 0.001", 0.00656),
          ("+ length weight 0.0025", 0.00639), ("+ Lovendal rule", 0.00596),
          ("+ length cap 22", 0.00585), ("+ MISSING penalty 0.15", 0.00478)]
fig = V.ladder([a for a, _ in LADDER], [b for _, b in LADDER]); plt.show()


## 11. Card detection

The experiment needs cropped label cards, so the detector is rebuilt here.

The trays photograph slightly blue and the paper labels are warm or neutral, which is a far more
reliable cue than brightness — some labels are pure white on a whitish tray. For those the tell
is **ink**: a real label carries strokes, so a pale region must show connected dark blobs of a
plausible size before it is accepted. Getting that veto wrong (rejecting on warmth alone) lost
cards in 50 of the 200 training plates.


In [ ]:
%%writefile mscat_cards.py
# -*- coding: utf-8 -*-
"""Label-card detection on 8192x5464 specimen tray photographs.

The trays photograph slightly blue; the paper labels are warm or neutral. That colour
gap is far more reliable than brightness alone, because some labels are pure white on a
whitish tray. For those, the tell is ink: a real label carries handwriting or print, so
we require connected dark strokes of a plausible size before accepting a pale region.

Everything is measured on a downscaled copy and mapped back, so a 45-megapixel plate
costs a fraction of a second.
"""
import numpy as np

try:
    import cv2
except Exception:                                   # pragma: no cover
    cv2 = None

# The left-hand strip of every plate holds the rig's mounting fixture, never a label.
FIXTURE_X = 0.262
DETECT_MAX_SIDE = 1400          # longest side of the working copy
MIN_AREA_FRAC = 4e-4            # of full plate area
MAX_AREA_FRAC = 0.10
MIN_ASPECT, MAX_ASPECT = 0.12, 8.0
MIN_STROKE_PX = 30              # mean ink-blob size, in ORIGINAL-resolution pixels


def _to_working(img, max_side=DETECT_MAX_SIDE):
    h, w = img.shape[:2]
    s = min(1.0, max_side / float(max(h, w)))
    if s >= 1.0:
        return img, 1.0
    return cv2.resize(img, (int(round(w * s)), int(round(h * s))),
                      interpolation=cv2.INTER_AREA), s


def _warmth(rgb):
    """R - B. Bluish tray goes negative, warm/neutral paper sits near or above zero."""
    return rgb[:, :, 0].astype(np.int16) - rgb[:, :, 2].astype(np.int16)


def _stroke_score(patch_gray):
    """Mean area of dark connected blobs inside a candidate, in working pixels."""
    if patch_gray.size == 0:
        return 0.0
    thr = max(30, int(np.percentile(patch_gray, 25)) - 12)
    ink = (patch_gray < thr).astype(np.uint8)
    if ink.sum() < 4:
        return 0.0
    n, _, stats, _ = cv2.connectedComponentsWithStats(ink, 8)
    if n <= 1:
        return 0.0
    areas = stats[1:, cv2.CC_STAT_AREA]
    areas = areas[areas >= 2]
    return float(areas.mean()) if areas.size else 0.0


def detect_cards(img_rgb, fixture_x=FIXTURE_X, debug=False):
    """Return card boxes [(x0, y0, x1, y1), ...] in ORIGINAL image coordinates.

    Boxes come back ordered by ink density, most written-on first, because the composite
    builder keeps only the most ink-rich crops.
    """
    if cv2 is None:
        raise RuntimeError("OpenCV is required for card detection")
    H, W = img_rgb.shape[:2]
    work, scale = _to_working(img_rgb)
    wh, ww = work.shape[:2]

    gray = cv2.cvtColor(work, cv2.COLOR_RGB2GRAY)
    warm = _warmth(work)

    # Adaptive tray statistics: the tray is whatever colour dominates the plate.
    tray_warm = float(np.median(warm))
    tray_val = float(np.median(gray))

    warm_mask = warm > (tray_warm + 6)
    bright_mask = gray > (tray_val + 18)
    mask = ((warm_mask | bright_mask).astype(np.uint8)) * 255

    k = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k, iterations=1)

    n, lab, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    plate_area = float(wh * ww)
    out = []
    for i in range(1, n):
        x, y, w, h, area = (stats[i, cv2.CC_STAT_LEFT], stats[i, cv2.CC_STAT_TOP],
                            stats[i, cv2.CC_STAT_WIDTH], stats[i, cv2.CC_STAT_HEIGHT],
                            stats[i, cv2.CC_STAT_AREA])
        if not (MIN_AREA_FRAC * plate_area <= area <= MAX_AREA_FRAC * plate_area):
            continue
        if h == 0 or not (MIN_ASPECT <= w / float(h) <= MAX_ASPECT):
            continue
        if (x + w / 2.0) / ww < fixture_x:          # mounting fixture, never a label
            continue

        patch = gray[y:y + h, x:x + w]
        stroke = _stroke_score(patch)
        # A pale region with no plausible ink is tray, not a label. Only pale ones are
        # asked to prove themselves; clearly warm paper is accepted on colour alone.
        pale = float(np.median(warm[y:y + h, x:x + w])) <= (tray_warm + 6)
        if pale and stroke * (1.0 / max(scale, 1e-6)) ** 2 < MIN_STROKE_PX:
            continue

        ink = float((patch < max(30, np.percentile(patch, 25) - 12)).mean())
        box = (int(x / scale), int(y / scale),
               int(min(W, (x + w) / scale)), int(min(H, (y + h) / scale)))
        out.append((box, ink))

    out.sort(key=lambda t: -t[1])
    boxes = [b for b, _ in out]
    return (boxes, mask, scale) if debug else boxes


def make_composite(img_rgb, boxes, keep=3, max_w=760, max_h=1520, pad=6):
    """Stack the most ink-rich crops into ONE image.

    Composites beat individual crops on this task (0.92/0.84 exact vs 0.90/0.86) at about
    a third of the token cost, because the reader sees every card of a specimen at once
    and can tell a date card from a locality card by context.
    """
    if not boxes:
        return None
    crops = []
    for (x0, y0, x1, y1) in boxes[:keep]:
        c = img_rgb[max(0, y0 - pad):y1 + pad, max(0, x0 - pad):x1 + pad]
        if c.size:
            crops.append(c)
    if not crops:
        return None

    w = min(max_w, max(c.shape[1] for c in crops))
    scaled = []
    for c in crops:
        s = w / float(c.shape[1])
        scaled.append(cv2.resize(c, (w, max(1, int(round(c.shape[0] * s)))),
                                 interpolation=cv2.INTER_AREA))
    total_h = sum(c.shape[0] for c in scaled)
    if total_h > max_h:                              # keep under the API downscale limit
        s = max_h / float(total_h)
        w = max(1, int(round(w * s)))
        scaled = [cv2.resize(c, (w, max(1, int(round(c.shape[0] * s)))),
                             interpolation=cv2.INTER_AREA) for c in scaled]
        total_h = sum(c.shape[0] for c in scaled)

    canvas = np.full((total_h, w, 3), 255, dtype=np.uint8)
    y = 0
    for c in scaled:
        canvas[y:y + c.shape[0], :c.shape[1]] = c
        y += c.shape[0]
    return canvas


In [ ]:
import mscat_cards as C

def find_images():
    for pat in ("/kaggle/input/**/*.jpeg", "/kaggle/input/**/*.jpg", "/kaggle/input/**/*.JPG"):
        fs = glob.glob(pat, recursive=True)
        if fs:
            return {os.path.basename(f): f for f in fs}
    return {}

IMAGES = find_images()
TRAIN_CSV = next((p for p in glob.glob("/kaggle/input/**/train.csv", recursive=True)), None)
print(f"images found : {len(IMAGES)}")
print(f"train.csv    : {TRAIN_CSV}")


## 12. The experiment — TrOCR fine-tuned on 200 plates

**The question.** The winning sheet came from vision-model reading, which is expensive and not
reproducible in a kernel. Would conventional supervised OCR, trained only on what the
competition ships, do the same job? And — separately — is a trained model's own sequence
likelihood a better confidence signal than the hand-built risk model in §8?

**The setup.** Detect cards on each labelled plate, composite them, fine-tune
`microsoft/trocr-base-handwritten` on (composite → `verbatimLocality`), hold out 40 plates, and
score the held-out split with the *same* AURC used by the leaderboard. Locality only: it is the
harder field and roughly three quarters of the score.

**Stated up front:** 160 training plates is a very small corpus for an OCR fine-tune, so I
expect this to lose badly to the shipped sheet. The experiment is worth running because *how* it
loses is informative — and because the confidence comparison is a fair test either way.


In [ ]:
%%writefile mscat_train.py
# -*- coding: utf-8 -*-
"""GPU experiment: fine-tune TrOCR on the 200 labelled specimens.

This is the *journal* arm of the notebook, not the submission arm. The point is to put a
number on a question the leaderboard cannot answer on its own: given only the labelled
data the competition ships, how far does a conventionally trained OCR model get, and is
its own sequence likelihood a better confidence signal than a hand-built risk model?

Everything is time-budgeted. The submission is written before this module is imported, so
a training failure can never cost a submission.
"""
import math
import os
import time

import numpy as np

try:
    import torch
    from torch.utils.data import Dataset, DataLoader
except Exception:                                    # pragma: no cover
    torch = None
    Dataset = object

TROCR_BASE = "microsoft/trocr-base-handwritten"
TROCR_SMALL = "microsoft/trocr-small-handwritten"

# The TrOCR repos ship only slow-tokenizer files. transformers 4.x converts them on the
# fly; 5.x removed that path and raises. The handwritten checkpoints use RoBERTa's
# vocabulary unchanged, so loading that tokenizer directly is an exact substitution.
TOKENIZER_FALLBACK = "roberta-base"


class _ComposedProcessor:
    """Stand-in for TrOCRProcessor, assembled from parts that load on every version."""

    def __init__(self, image_processor, tokenizer):
        self.image_processor = image_processor
        self.tokenizer = tokenizer

    def __call__(self, images=None, **kw):
        return self.image_processor(images=images, **kw)

    def batch_decode(self, *a, **k):
        return self.tokenizer.batch_decode(*a, **k)

    def save_pretrained(self, d):
        self.image_processor.save_pretrained(d)
        self.tokenizer.save_pretrained(d)


def load_processor(model_name, verbose=True):
    """TrOCRProcessor when it works, an equivalent composition when it does not."""
    from transformers import TrOCRProcessor
    try:
        return TrOCRProcessor.from_pretrained(model_name)
    except Exception as exc:
        from transformers import AutoImageProcessor, AutoTokenizer
        if verbose:
            print(f"    TrOCRProcessor unavailable ({type(exc).__name__}); "
                  f"composing image processor + {TOKENIZER_FALLBACK} tokenizer")
        return _ComposedProcessor(AutoImageProcessor.from_pretrained(model_name),
                                  AutoTokenizer.from_pretrained(TOKENIZER_FALLBACK))


# --------------------------------------------------------------- data
class CropDataset(Dataset):
    """(card crop, verbatim string) pairs, with light augmentation for the tiny train set."""

    def __init__(self, items, processor, max_len=64, augment=False):
        self.items = items
        self.processor = processor
        self.max_len = max_len
        self.augment = augment

    def __len__(self):
        return len(self.items)

    def _aug(self, img):
        import cv2
        h, w = img.shape[:2]
        ang = np.random.uniform(-3.0, 3.0)                       # labels sit slightly askew
        M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, 1.0)
        img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
        a = np.random.uniform(0.85, 1.15)                        # exposure varies per plate
        b = np.random.uniform(-18, 18)
        img = np.clip(img.astype(np.float32) * a + b, 0, 255).astype(np.uint8)
        if np.random.rand() < 0.25:
            img = cv2.GaussianBlur(img, (3, 3), 0)
        return img

    def __getitem__(self, i):
        img, text = self.items[i]
        img = np.asarray(img)
        if self.augment:
            img = self._aug(img)
        px = self.processor(images=img, return_tensors="pt").pixel_values[0]
        tok = _tokenizer(self.processor)
        ids = tok(text, padding="max_length", truncation=True,
                  max_length=self.max_len).input_ids
        ids = [t if t != tok.pad_token_id else -100 for t in ids]
        return {"pixel_values": px, "labels": torch.tensor(ids)}


def _tokenizer(processor):
    """TrOCRProcessor exposes its tokenizer under different names across versions."""
    for attr in ("tokenizer", "text_processor"):
        t = getattr(processor, attr, None)
        if t is not None:
            return t
    raise AttributeError("no tokenizer on processor")


def collate(batch):
    return {"pixel_values": torch.stack([b["pixel_values"] for b in batch]),
            "labels": torch.stack([b["labels"] for b in batch])}


# --------------------------------------------------------------- train
def train(items, val_items, budget_s, model_name=TROCR_BASE, out_dir="trocr_out",
          batch_size=4, lr=4e-5, max_epochs=40, log_every=25, seed=0, verbose=True):
    """Fine-tune until the epoch budget or the wall-clock budget runs out.

    Returns a history dict; never raises on budget exhaustion, which is the normal way it
    finishes.
    """
    from transformers import VisionEncoderDecoderModel

    t0 = time.time()
    torch.manual_seed(seed)
    np.random.seed(seed)
    os.makedirs(out_dir, exist_ok=True)

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    processor = load_processor(model_name, verbose=verbose)
    model = VisionEncoderDecoderModel.from_pretrained(model_name).to(dev)

    tok = _tokenizer(processor)
    model.config.decoder_start_token_id = tok.cls_token_id or tok.bos_token_id
    model.config.pad_token_id = tok.pad_token_id
    model.config.eos_token_id = tok.sep_token_id or tok.eos_token_id
    model.config.vocab_size = model.config.decoder.vocab_size

    tr = DataLoader(CropDataset(items, processor, augment=True), batch_size=batch_size,
                    shuffle=True, collate_fn=collate, num_workers=0)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    amp = (dev == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=amp)

    hist = {"step": [], "loss": [], "epoch_loss": [], "elapsed": [],
            "stopped_on": None, "device": dev, "model": model_name}
    step = 0
    model.train()
    for ep in range(max_epochs):
        run, nb = 0.0, 0
        for batch in tr:
            if time.time() - t0 > budget_s:
                hist["stopped_on"] = f"time budget after {ep} epochs"
                break
            px = batch["pixel_values"].to(dev, non_blocking=True)
            lb = batch["labels"].to(dev, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=amp):
                loss = model(pixel_values=px, labels=lb).loss
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            run += float(loss.item())
            nb += 1
            step += 1
            if verbose and step % log_every == 0:
                print(f"    step {step:5d}  loss {run / max(nb, 1):.4f}  "
                      f"{time.time() - t0:6.0f}s")
            hist["step"].append(step)
            hist["loss"].append(float(loss.item()))
            hist["elapsed"].append(time.time() - t0)
        if nb:
            partial = " (partial - budget reached)" if hist["stopped_on"] else ""
            hist["epoch_loss"].append(run / nb)
            if verbose:
                print(f"  epoch {ep + 1:3d}  mean loss {run / nb:.4f}  "
                      f"elapsed {time.time() - t0:.0f}s{partial}")
        if hist["stopped_on"]:
            break
    else:
        hist["stopped_on"] = "max_epochs"

    model.save_pretrained(out_dir)
    processor.save_pretrained(out_dir)
    hist["total_s"] = time.time() - t0
    return model, processor, hist


# --------------------------------------------------------------- predict
@torch.no_grad() if torch else (lambda f: f)
def predict(model, processor, images, batch_size=8, max_len=64, num_beams=1):
    """Transcribe crops and return (texts, confidences).

    Confidence is the mean per-token log-probability of the emitted sequence, mapped to
    (0, 1]. That is the model's own belief, and the notebook tests it head-to-head against
    the hand-built risk model rather than assuming it is better.
    """
    dev = next(model.parameters()).device
    model.eval()
    tok = _tokenizer(processor)
    texts, confs = [], []
    for i in range(0, len(images), batch_size):
        chunk = [np.asarray(im) for im in images[i:i + batch_size]]
        px = processor(images=chunk, return_tensors="pt").pixel_values.to(dev)
        out = model.generate(px, max_length=max_len, num_beams=num_beams,
                             output_scores=True, return_dict_in_generate=True)
        seqs = out.sequences
        texts.extend(tok.batch_decode(seqs, skip_special_tokens=True))

        if getattr(out, "scores", None):
            # mean log-prob of the chosen token at each generated position
            lps = []
            for t, logits in enumerate(out.scores):
                lp = torch.log_softmax(logits.float(), dim=-1)
                nxt = seqs[:, t + 1] if t + 1 < seqs.shape[1] else seqs[:, -1]
                lps.append(lp.gather(1, nxt.unsqueeze(1)).squeeze(1))
            m = torch.stack(lps, 1)
            pad = tok.pad_token_id
            mask = (seqs[:, 1:1 + m.shape[1]] != pad).float()
            mean_lp = (m * mask).sum(1) / mask.sum(1).clamp(min=1)
            confs.extend(torch.exp(mean_lp).clamp(0, 1).tolist())
        else:
            confs.extend([0.5] * len(chunk))
    return texts, confs


In [ ]:
import mscat_train as TR

EXPERIMENT = {"status": "not started", "n_train": 0, "n_val": 0}
PAIRS = []

if not (IMAGES and TRAIN_CSV):
    EXPERIMENT["status"] = "skipped - competition images or train.csv not attached"
elif not HAS_GPU:
    EXPERIMENT["status"] = "skipped - no GPU in this session"
else:
    gt = read_csv(TRAIN_CSV)
    t_crop = time.time()
    CROP_CAP_S = 40 * 60          # cropping must not eat the training budget
    import cv2
    bad = 0
    for i, r in enumerate(gt):
        if time.time() - t_crop > CROP_CAP_S:
            print(f"  crop cap reached at {i}/{len(gt)}")
            break
        f = IMAGES.get(r["image_file"])
        if not f:
            continue
        try:
            raw = cv2.imread(f)               # returns None on an unreadable file
            if raw is None:
                bad += 1
                continue
            img = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
            comp = C.make_composite(img, C.detect_cards(img), keep=3)
        except Exception:
            bad += 1
            continue                          # one bad plate must not end the experiment
        if comp is not None:
            PAIRS.append((comp, clean_locality(r.get("verbatimLocality", "") or "MISSING")))
        if (i + 1) % 50 == 0:
            print(f"  cropped {i+1}/{len(gt)}  ({time.time()-t_crop:.0f}s)")
    print(f"usable (crop, text) pairs: {len(PAIRS)}  in {time.time()-t_crop:.0f}s"
          f"  ({bad} unreadable)")
    EXPERIMENT["status"] = "ready" if len(PAIRS) >= 50 else f"skipped - only {len(PAIRS)} pairs"

print("experiment:", EXPERIMENT["status"])


In [ ]:
HIST = None
if EXPERIMENT["status"] == "ready":
    rs = np.random.RandomState(0)
    idx = rs.permutation(len(PAIRS))
    n_val = max(20, int(0.2 * len(PAIRS)))
    val_items = [PAIRS[i] for i in idx[:n_val]]
    trn_items = [PAIRS[i] for i in idx[n_val:]]
    EXPERIMENT.update(n_train=len(trn_items), n_val=len(val_items))

    budget = max(60.0, remaining() - RESERVE_S)
    print(f"training on {len(trn_items)} plates, validating on {len(val_items)}")
    print(f"wall-clock budget for training: {budget/3600:.2f} h\n")
    try:
        MODEL, PROC, HIST = TR.train(trn_items, val_items, budget_s=budget,
                                     model_name=TR.TROCR_BASE, out_dir="trocr_locality",
                                     batch_size=4, lr=4e-5, max_epochs=60)
        EXPERIMENT["status"] = "trained"
        print(f"\nstopped on: {HIST['stopped_on']}   total {HIST['total_s']/3600:.2f} h")
    except Exception as exc:
        EXPERIMENT["status"] = f"failed: {type(exc).__name__}: {exc}"
        print("training failed:", EXPERIMENT["status"])
else:
    print("training not run -", EXPERIMENT["status"])


## 13. Experiment results

Three numbers decide whether the experiment says anything: the fine-tune's transcription
accuracy, its AURC when ranked by **its own likelihood**, and its AURC when ranked by the
**hand-built risk model** over the exact same text. The third is the one I care about — it
isolates ranking from reading.


In [ ]:
RESULTS = {}
if EXPERIMENT["status"] == "trained":
    texts, confs = TR.predict(MODEL, PROC, [im for im, _ in val_items], batch_size=8)
    truth = [t for _, t in val_items]
    errs = [ned_cards(p, g) for p, g in zip(texts, truth)]

    risk_conf = R.to_confidence(R.locality_risk(texts))
    RESULTS = {
        "n_val": len(truth),
        "mean_NED": float(np.mean(errs)),
        "exact_match": float(np.mean([e == 0 for e in errs])),
        "AURC_model_confidence": aurc(errs, confs),
        "AURC_risk_model": aurc(errs, risk_conf),
        "AURC_random": float(np.mean([aurc(errs, np.random.RandomState(s).rand(len(errs)))
                                      for s in range(50)])),
    }
    for k, v in RESULTS.items():
        print(f"  {k:24} {v:.4f}" if isinstance(v, float) else f"  {k:24} {v}")

    if HIST and HIST["epoch_loss"]:
        V.style()
        f, ax = plt.subplots(figsize=(7.6, 3.8))
        ax.plot(range(1, len(HIST["epoch_loss"]) + 1), HIST["epoch_loss"],
                lw=2.0, color=V.S1)
        V._frame(ax, "TrOCR fine-tune: training loss", "epoch", "mean cross-entropy")
        plt.show()
else:
    print("no results -", EXPERIMENT["status"])
    print("\nThe submission in section 3 is unaffected; it was written before this ran.")


### Reading the result

Whatever the numbers say, one comparison is structural and worth stating in advance.
`AURC_model_confidence` and `AURC_risk_model` are computed over **identical text** — the
fine-tune's own output. Any gap between them is caused purely by ordering, and it is a direct
test of the notebook's central claim.

There is prior evidence for the answer. An external reader's own confidence was tried as a
ranking feature on the real test set and scored **0.00809**, far worse than the hand-built risk
model at the same text accuracy. A model's likelihood is calibrated for *its own* error
distribution, not for normalised edit distance against a curator's transcription — and those
are not the same thing.


## 14. What did not work

Recorded because these are the natural next ideas, and every one is worse than doing nothing.

| idea | LB | why it fails |
|---|---|---|
| 3-reader majority vote over the arbiter | 0.01075 | 2-1 votes overturn full-resolution verdicts and delete real localities |
| promote bare 2-3 digit cards to dates | 0.00992 | they are catalogue numbers in this collection |
| percentile-rank fusion instead of additive risk | 0.00965 | discards the magnitude of each signal |
| external reader's own confidence as a feature | 0.00809 | calibrated for a different error distribution |
| 5th-pass adjudication of the 700 riskiest rows | 0.00665 | extra reads on already-safe rows are noise, not evidence |
| MISSING penalty applied to **dates** | 0.00540 | absent dates genuinely are near-certain |
| gazetteer used to *repair text* instead of to rank | neutral | corrections and corruptions cancel out |

Two patterns run through the whole list.

**Extra reads may feed risk, but must never overwrite text.** The arbiter sees the
full-resolution crop; a majority of lower-resolution readers does not outrank it.

**The two fields are not the same problem.** A `MISSING` locality is usually a miss, so it
belongs mid-ranking. A `MISSING` date means the card genuinely has no date on it, which is
unambiguous, so it belongs at the top. Applying the winning locality trick to dates cost
0.00478 → 0.00540. Symmetry was the wrong instinct here.


## 15. Closing — and a last check on the artefact

The submission was written in §3 and nothing since has been allowed to touch it. This re-runs
the same validation on the file as it exists now, so the notebook ends by proving the thing it
was supposed to produce is still intact.


In [ ]:
print("final artefact check")
SUB2, DC2, LC2, OK2 = validate()

same = os.path.exists(SUBMISSION) and len(SUB2) == N_TEST
print(f"\nmode              : {SUB_MODE}")
print(f"all checks passed : {OK2}")
print(f"experiment        : {EXPERIMENT['status']}")
if RESULTS:
    print(f"  fine-tune mean NED {RESULTS['mean_NED']:.4f} "
          f"| AURC own-confidence {RESULTS['AURC_model_confidence']:.4f} "
          f"| AURC risk-model {RESULTS['AURC_risk_model']:.4f}")
print(f"total elapsed     : {elapsed()/3600:.2f} h of {TOTAL_BUDGET_S/3600:.1f} h")
print(f"\n{SUBMISSION} is ready to submit.")
